In [1]:
import os
import re
import io
import cv2
import math
import time
import random
import shutil
import errno
import tarfile
import logging
import warnings
import collections

import numpy as np
import matplotlib.pyplot as plt


from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

import hashlib
import requests

from tqdm import tqdm

import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.nn.functional as F

from torchvision.datasets.utils import download_and_extract_archive
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks

from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import (
    LastLevelMaxPool,
    FeaturePyramidNetwork
)
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as FT
from torchvision.transforms import v2 as T

try:
    import lightning as L
except:
    import lightning as L

from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from torchmetrics.detection.mean_ap import MeanAveragePrecision
import os
import sys
sys.path.append(os.path.join(os.getcwd(), *tuple(['..'])))
from features import build_features
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams['axes.facecolor'] = 'lightgray'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'STIXGeneral'


In [37]:
EXPERIMENT_DIR  = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/ViTMaskRCNN"
os.makedirs(os.path.join(EXPERIMENT_DIR,"experiment"), exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR,"experiment/training"), exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR,"experiment/dataset"), exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR,"experiment/model"), exist_ok=True)
EXPERIMENT_DIR ="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/ViTMaskRCNN/experiment"

In [2]:
NUM_LAYER = 12
PATCH_SIZE = 16
HIDDEN_DIM = 768
NUM_CLASSES = 2
IMG_SIZE = 1024
INFERENCE_SAMPLE = 3
THRESHOLD_DETECTION = 0.75
THRESHOLD_SEGMENTATION = 0.5

In [3]:
MAX_EPOCH               = 5  #312
BATCH_SIZE              = 2  #17
EARLY_STOPPING_PATIENCE = 1 #31

In [4]:
LEARNING_RATE    = 1 / 62.8318 * (math.pi / math.e)
MOMENTUM         = 1 / 9.41149 * (math.pi * math.e)
WEIGHT_DECAY     = 1 / 12345.6 * (math.pi + math.e)
REDUCE_LR_FACTOR = 1 / 0.56789 * (math.pi - math.e)

In [5]:
MILESTONES = 1. / math.sqrt(MAX_EPOCH) * (
    np.array(
        [m for m in range(1, int(math.sqrt(MAX_EPOCH)))]
    )
)

In [6]:
METRIC_TO_MONITOR = "val_loss"
METRIC_MODE       = "min"
SEED = int(np.random.randint(2147483647))
print(f"Random seed: {SEED}")

Random seed: 1328446351


In [7]:
class IntermediateLayerGetter(nn.ModuleDict):
    _version = 2
    __annotations__ = {
        "return_layers",
    }

    def __init__(self, model, return_layers):
        if not set(return_layers).issubset(
            [name for name, _ in model.named_children()]
        ):
            raise ValueError("return_layers are not present in model")
        orig_return_layers = return_layers
        return_layers = {str(k): str(v) for k, v in return_layers.items()}
        layers = collections.OrderedDict()
        for name, module in model.named_children():
            layers[name] = module
            if name in return_layers:
                del return_layers[name]
            if not return_layers:
                break

        super().__init__(layers)
        self.return_layers = orig_return_layers

        self.C = HIDDEN_DIM
        self.H = self.W = IMG_SIZE // PATCH_SIZE

    def forward(self, x):
        out = collections.OrderedDict()
        idx = 0
        for name, module in self.items():
            x = module(x)
            if name in self.return_layers:
                out_name = self.return_layers[name]
                N = x.shape[0]
                out[out_name] = F.interpolate(
                    F.instance_norm(
                        x.permute(0, 2, 1).reshape(N, self.C, self.H, self.W)
                    ),
                    scale_factor=4 / (2**idx),
                    mode="bilinear",
                )
                idx += 1
        return out

In [19]:
class BackboneWithFPN(nn.Module):
    def __init__(
        self,
        backbone,
        return_layers,
        in_channels_list,
        out_channels,
        extra_blocks=None,
        norm_layer=nn.BatchNorm2d,
    ):
        super().__init__()

        if extra_blocks is None:
            extra_blocks = LastLevelMaxPool()

        self.backbone = backbone
        print([x for x in self.backbone.encoder.layers])
        self.body = IntermediateLayerGetter(
            self.backbone.encoder.layers,
            #self.backbone.blocks,
            return_layers=return_layers,
        )
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=out_channels,
            extra_blocks=extra_blocks,
            norm_layer=norm_layer,
        )
        self.out_channels = out_channels

    def forward(self, x):
        x = self.backbone._process_input(x)
        x = x + self.backbone.patch_embed(x)
        x = self.backbone.encoder.dropout(x)
        x = self.body(x)
        x = self.fpn(x)
        return x

In [23]:
import torch
import torch.nn.functional as F
from torchvision.models.vision_transformer import vit_b_16, ViT_B_16_Weights
from torchvision.models.detection.backbone_utils import BackboneWithFPN

# Constants
NUM_LAYER = 12
HIDDEN_DIM = 768  # ViT-Base hidden size
IMG_SIZE = 1024
PATCH_SIZE = 16
PATCH_GRID = IMG_SIZE // PATCH_SIZE  # 64

# Load pretrained ViT
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

# Interpolate positional embeddings
cls_token = vit.encoder.pos_embedding[:, :1, :]
patch_pos = vit.encoder.pos_embedding[:, 1:, :]
patch_pos = patch_pos.reshape(1, 14, 14, -1).permute(0, 3, 1, 2)
patch_pos = F.interpolate(patch_pos, size=(PATCH_GRID, PATCH_GRID), mode='bicubic', align_corners=False)
patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, PATCH_GRID * PATCH_GRID, -1)
vit.encoder.pos_embedding = torch.nn.Parameter(torch.cat([cls_token, patch_pos], dim=1))




In [133]:
out.shape

torch.Size([1, 1000])

In [134]:
backbone = backbone_utils.resnet_fpn_backbone(
            #'resnext101_32x8d',
            'resnet152',
            #'resnet152',
            pretrained=True,
            trainable_layers=3
        )

In [136]:
x = torch.randn(1, 3, 1024, 1024)
out = backbone(x)
out

OrderedDict([('0',
              tensor([[[[ 0.7530,  0.7013,  0.5554,  ...,  0.7607,  1.0358,  0.2244],
                        [ 1.2945,  1.0581,  1.1789,  ...,  0.7306,  1.2034,  1.1084],
                        [ 1.3014,  1.2635,  1.0471,  ...,  0.8599,  1.2279,  1.0775],
                        ...,
                        [ 1.1623,  1.0235,  0.7712,  ...,  0.6397,  0.7748,  0.5994],
                        [ 1.1624,  0.9577,  0.7909,  ...,  0.7836,  0.8892,  0.7397],
                        [ 0.9150,  0.9725,  0.8532,  ...,  0.5632,  0.5854,  0.5955]],
              
                       [[-0.2289,  0.9718,  1.0452,  ...,  1.2507,  1.0974,  0.7669],
                        [-0.1545,  0.7074,  0.6259,  ...,  0.6166,  0.5519,  0.5648],
                        [-0.3126,  0.2775,  0.5341,  ...,  0.6117,  0.4264,  0.2712],
                        ...,
                        [ 0.3024,  0.9979,  1.0161,  ...,  0.7724,  0.6464,  0.3211],
                        [ 0.4685,  1.0203,  0.9

In [54]:
class ViTMaskRCNN(L.LightningModule):
    def __init__(self):
        super().__init__()

        self.batch_size = BATCH_SIZE
        self.lr = LEARNING_RATE
        self.max_epoch = MAX_EPOCH
        self.lr_now = self.lr * 1e3

        self.backbone = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        self.backbone.encoder.pos_embedding = torch.nn.Parameter(
            self.backbone.encoder.pos_embedding[:, 1:, :]
        )
        del self.backbone.class_token, self.backbone.encoder.ln
        self.backbone = BackboneWithFPN(
            self.backbone,
            {
                f"encoder_layer_{(NUM_LAYER - 1) - l}": str(
                    ((NUM_LAYER - 1) - l - 2) // 3
                )
                for l in range(9, -1, -3)
            },
            [HIDDEN_DIM] * 4,
            1024,
        )

        self.mask_rcnn = maskrcnn_resnet50_fpn_v2(
            weights=MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1
        )
        self.mask_rcnn.backbone = self.backbone
        self.mask_rcnn.transform.min_size = [IMG_SIZE]
        self.mask_rcnn.transform.max_size = IMG_SIZE
        self.mask_rcnn.transform.fixed_size = (IMG_SIZE, IMG_SIZE)
        self.mask_rcnn.num_classes = NUM_CLASSES
        self.mask_rcnn.roi_heads.box_predictor.cls_score = nn.Linear(
            1024,
            NUM_CLASSES,
        )
        self.mask_rcnn.roi_heads.box_predictor.bbox_pred = nn.Linear(
            1024,
            4 * NUM_CLASSES,
        )
        self.mask_rcnn.roi_heads.mask_predictor.mask_fcn_logits = nn.Conv2d(
            1024, NUM_CLASSES, 1,
        )

        self.test_mAP_box = MeanAveragePrecision(
            box_format="xyxy",
            iou_type="bbox",
        )
        self.test_mAP_seg = MeanAveragePrecision(
            iou_type="segm",
        )

        self.automatic_optimization = False

        self.train_loss = list()
        self.val_loss = list()

        self.train_loss_recorder = AvgMeter()
        self.val_loss_recorder = AvgMeter()

        self.sanity_check_counter = 1

    def forward(self, x, y=None):
        return self.mask_rcnn(x, y)

    def training_step(self, batch, batch_nb):
        x, y = batch
        x = list(image.to(self.device) for image in x)
        y = [
            {
                k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                for k, v in t.items()
            }
            for t in y
        ]
        loss = self(x, y)
        loss = sum(l for l in loss.values())

        opt = self.optimizers()
        opt.zero_grad()
        self.manual_backward(loss)
        opt.step()

        self.log("train_loss", loss, prog_bar=True)

        self.train_loss_recorder.update(loss.data)

    def on_train_epoch_end(self):
        sch = self.lr_schedulers()
        sch.step()

        self.train_loss.append(
            self.train_loss_recorder.show().data.cpu().numpy(),
        )
        self.train_loss_recorder = AvgMeter()

    def validation_step(self, batch, batch_nb):
        x, y = batch
        x = list(image.to(self.device) for image in x)
        y = [
            {
                k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                for k, v in t.items()
            }
            for t in y
        ]
        self.mask_rcnn.train()
        loss = self(x, y)
        loss = sum(l for l in loss.values()).detach()

        if self.sanity_check_counter == 0:
            self.log("val_loss", loss, prog_bar=True)
            self.val_loss_recorder.update(loss.data)

    def on_validation_epoch_end(self):
        if self.sanity_check_counter == 0:
            loss = self.val_loss_recorder.show().data.cpu().numpy()
            lr_now_ = self.optimizers().param_groups[0]["lr"]
            if self.lr_now != lr_now_:
                self.lr_now = lr_now_
                str_report = f"[{MODEL_NAME}] Learning Rate Changed: {lr_now_}"
                str_report += f"- Epoch: {self.current_epoch}"
                print(str_report)
            self.val_loss.append(loss)
            self.val_loss_recorder = AvgMeter()
        else:
            self.sanity_check_counter -= 1

    def test_step(self, batch, batch_nb):
        x, y = batch
        x = list(image.to(self.device) for image in x)
        y = [
            {
                k: v.to(self.device) if isinstance(v, torch.Tensor) else v
                for k, v in t.items()
            }
            for t in y
        ]
        y_hat = self(x)

        pred_box = list()
        target_box = list()

        pred_seg = list()
        target_seg = list()

        for b in range(len(y_hat)):
            try:
                pred_box.append(
                    dict(
                        boxes=y_hat[b]["boxes"],
                        scores=y_hat[b]["scores"],
                        labels=y_hat[b]["labels"].int(),
                    )
                )
            except:
                pred_box.append(
                    dict(
                        boxes=torch.tensor(
                            [[0.0, 0.0, 0.0, 0.0]],
                        ).to(self.device),
                        scores=torch.tensor([0.0]).to(self.device),
                        labels=torch.tensor([0.0]).to(self.device).int(),
                    )
                )

            try:
                pred_seg.append(
                    dict(
                        masks=(y_hat[b]["masks"] > THRESHOLD_SEGMENTATION)
                        .squeeze(1)
                        .to(torch.bool),
                        scores=y_hat[b]["scores"],
                        labels=y_hat[b]["labels"].int(),
                    )
                )
            except:
                pred_seg.append(
                    dict(
                        masks=torch.zeros(1, x[b].shape[-2], x[b].shape[-1])
                        .to(torch.bool)
                        .to(self.device),
                        scores=torch.tensor([0.0]).to(self.device),
                        labels=torch.tensor([0.0]).to(self.device).int(),
                    )
                )

            target_box.append(
                dict(
                    boxes=y[b]["boxes"],
                    labels=y[b]["labels"].int(),
                )
            )

            target_seg.append(
                dict(
                    masks=(y[b]["masks"] > THRESHOLD_SEGMENTATION)
                    .squeeze(1)
                    .to(torch.bool),
                    labels=y[b]["labels"].int(),
                )
            )

        self.test_mAP_box.update(pred_box, target_box)
        mAP_box = self.test_mAP_box.compute()["map"].data.detach().cpu()

        self.test_mAP_seg.update(pred_seg, target_seg)
        mAP_seg = self.test_mAP_seg.compute()["map"].data.detach().cpu()

        self.log("[BOX]_test_mAP@0.5:0.95", mAP_box, prog_bar=True, logger=True)
        self.log("[SEG]_test_mAP@0.5:0.95", mAP_seg, prog_bar=True, logger=True)

    def on_train_end(self):

        # Loss
        loss_img_file = f"experiment/training/{MODEL_NAME}_loss_plot.png"
        plt.plot(self.train_loss, color="r", label="train")
        plt.plot(self.val_loss, color="b", label="validation")
        plt.title("Loss Curves")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.savefig(loss_img_file)
        plt.clf()
        img = cv2.imread(loss_img_file)
        cv2_imshow(img)

    def train_dataloader(self):
        return data.DataLoader(
            TrainDataset,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=2,
            persistent_workers=True,
        )

    def val_dataloader(self):
        return data.DataLoader(
            ValDataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=2,
            persistent_workers=True,
        )

    def test_dataloader(self):
        return data.DataLoader(
            TestDataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=2,
            persistent_workers=True,
        )

    def configure_optimizers(self):
        optimizer = optim.SGD(
            self.parameters(),
            lr=self.lr,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            nesterov=True,
        )

        lr_scheduler = {
            "scheduler": optim.lr_scheduler.MultiStepLR(
                optimizer,
                milestones=[int(self.max_epoch * ms) for ms in MILESTONES],
                gamma=REDUCE_LR_FACTOR,
            ),
            "name": "lr_scheduler",
        }

        return [optimizer], [lr_scheduler]

In [55]:
MODEL_NAME = ViTMaskRCNN.__name__
MODEL = ViTMaskRCNN
BEST_MODEL_PATH = os.path.join(
    EXPERIMENT_DIR,
    "model",
    f"{MODEL_NAME}_best.ckpt",
)

In [56]:
MODEL_NAME

'ViTMaskRCNN'

In [57]:
class AvgMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.scores = list()

    def update(self, val):
        self.scores.append(val)

    def show(self):
        scores = torch.stack(self.scores)
        return torch.mean(scores)

In [58]:
def _train_loop():
    seed_everything(SEED, workers=True)

    print(MODEL_NAME)
    model = MODEL()

    callbacks = list()

    checkpoint = ModelCheckpoint(
        monitor=METRIC_TO_MONITOR,
        dirpath=f"{EXPERIMENT_DIR}/model",
        mode=METRIC_MODE,
        filename=f"{MODEL_NAME}_best",
    )
    callbacks.append(checkpoint)

    early_stopping = EarlyStopping(
        monitor=METRIC_TO_MONITOR,
        min_delta=0.00,
        patience=EARLY_STOPPING_PATIENCE,
        verbose=False,
        mode=METRIC_MODE,
    )
    callbacks.append(early_stopping)

    if os.path.exists(BEST_MODEL_PATH):
        ckpt_path = BEST_MODEL_PATH
    else:
        ckpt_path = None

    trainer = Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=MAX_EPOCH,
        logger=False,
        callbacks=callbacks,
        log_every_n_steps=5,
    )
    trainer.fit(model, ckpt_path=ckpt_path)

_train_loop()

Seed set to 885614065


ViTMaskRCNN


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type                 | Params
------------------------------------------------------
0 | backbone     | BackboneWithFPN      | 127 M 
1 | mask_rcnn    | MaskRCNN             | 146 M 
2 | test_mAP_box | MeanAveragePrecision | 0     
3 | test_mAP_seg | MeanAveragePrecision | 0     
------------------------------------------------------
146 M     Trainable params
0         Non-trainable params
146 M     Total params
586.012   Total estimated model params size (MB)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

AssertionError: Wrong image height! Expected 224 but got 1024!

In [34]:
dataset_train_location = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train_corrected/"
dataset_val_location = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val_corrected/"

In [51]:
TrainDataset = build_features.LBD_Dataset(dataset_train_location, T.Compose([T.ToTensor()]), ['image','mask'])
ValDataset = build_features.LBD_Dataset(dataset_val_location, T.Compose([T.ToTensor()]),['image','mask'])

In [52]:
collate_fn=lambda x: tuple(zip(*x))

In [ ]:
train_data_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=train_config['batch_size'], shuffle=True, num_workers=4,
            collate_fn=collate_fn)

val_data_loader = torch.utils.data.DataLoader(
            val_dataset, batch_size=train_config['batch_size'], shuffle=False, num_workers=4,
            collate_fn=collate_fn)